# Chapter 13 — Java (translated to Python concepts)

Each Java question is answered first in Java terms, then with its Python counterpart. Most of them have one.

| Java | Python |
|---|---|
| `private` constructor | `__new__` guards / factory classmethods / `@final` |
| `try/finally` | `try/finally`, context managers |
| `final` | `typing.Final`, `@typing.final` (static checks only) |
| `finalize()` | `__del__` / `weakref.finalize` |
| generics (type erasure) | `typing.Generic` (also erased at runtime!) |
| `HashMap` / `TreeMap` / `LinkedHashMap` | `dict` / sorted map / `dict` + `move_to_end` (`OrderedDict`) |
| reflection | `getattr`, `inspect`, `type()` |
| lambdas + streams | `lambda`, comprehensions, `filter`/`map`/`sum` |

In [1]:
import inspect
import random
import weakref
from collections import OrderedDict
from typing import Final, Generic, TypeVar, final, get_args

## 13.1 Private Constructor
In Java, a private constructor means only the class itself (and inner classes) can instantiate it. That's the basis of the singleton and factory patterns, and of classes that must not be subclassed from outside. Python has no access modifiers, but the same intent can be enforced.

In [2]:
class Connection:
    _token = object()

    def __init__(self, dsn: str, *, _token: object = None) -> None:
        if _token is not Connection._token:
            raise TypeError("use Connection.open()")
        self.dsn = dsn

    @classmethod
    def open(cls, dsn: str) -> "Connection":   # the only public way to construct
        return cls(dsn, _token=cls._token)


assert Connection.open("db://x").dsn == "db://x"
try:
    Connection("db://x"); raise AssertionError
except TypeError as e:
    print("blocked:", e)

blocked: use Connection.open()


## 13.2 Return from Finally
Does `finally` run if the `try` block has a `return`? **Yes.** It is skipped only if the VM exits or the thread is killed. Python behaves the same way. A `return` *inside* `finally` overrides the original return and swallows exceptions. Python 3.14 (PEP 765) emits a `SyntaxWarning` for it, so don't do it. We build that function with `exec` so the warning doesn't clutter this notebook.

In [3]:
trace: list[str] = []


def f() -> str:
    try:
        return "from try"
    finally:
        trace.append("finally ran")


assert f() == "from try" and trace == ["finally ran"]

import warnings
ns: dict = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", SyntaxWarning)
    exec("def g():\n    try:\n        raise ValueError('lost')\n    finally:\n        return 'finally wins'\n", ns)
assert ns["g"]() == "finally wins"  # the exception is silently swallowed - avoid!
print("13.2 ok")

13.2 ok


## 13.3 Final, Finally, Finalize
- `final`: a variable can't be reassigned, a method can't be overridden, a class can't be subclassed. → Python: `typing.Final` and `@typing.final`. These are enforced by type checkers (mypy/pyright), **not** at runtime. `__init_subclass__` can enforce the rule at runtime.
- `finally`: a block that always runs after `try`/`catch`. → Same in Python.
- `finalize()`: called by the GC before it reclaims an object. Deprecated in Java 9. → `__del__`, or better, `weakref.finalize`.

In [4]:
MAX: Final = 10


@final
class Sealed:
    def __init_subclass__(cls, **kw):  # runtime enforcement of 'final class'
        raise TypeError("Sealed is final")


try:
    class Sub(Sealed): ...
except TypeError as e:
    print("final class:", e)


class Resource: ...


events: list[str] = []
r = Resource()
weakref.finalize(r, events.append, "finalized")
del r
assert events == ["finalized"]
print("13.3 ok")

final class: Sealed is final
13.3 ok


## 13.4 Generics vs. Templates
C++ templates generate a new copy of the code for each type, so `List<int>` and `List<string>` are separate classes. Java generics use **type erasure**: one class at runtime, and `List<String>` becomes `List`. So Java can't use primitive type arguments, and static fields are shared across instantiations. Python's `Generic[T]` is also erased at runtime. It's purely for static checkers.

In [5]:
T = TypeVar("T")


class Box(Generic[T]):
    shared_count = 0  # like a Java static field: shared across Box[int], Box[str]

    def __init__(self, item: T) -> None:
        self.item = item
        Box.shared_count += 1


bi, bs = Box[int](1), Box[str]("x")
assert type(bi) is type(bs) is Box          # erasure: one runtime class
assert Box.shared_count == 2
assert get_args(Box[int]) == (int,)         # parameter only visible on the alias
print("13.4 ok")

13.4 ok


## 13.5 TreeMap, HashMap, LinkedHashMap
- `HashMap`: `O(1)` operations, no ordering. → `dict` (which also keeps insertion order in Python 3.7+).
- `TreeMap`: `O(log n)` operations, sorted by key. → a sorted container (see Chapter 12's `SortedMap`, or `sortedcontainers`).
- `LinkedHashMap`: `O(1)` operations, keeps insertion order or access order, useful for LRU caches. → `dict` / `OrderedDict.move_to_end`.

In [6]:
keys = [5, 1, 9, 3]
hash_map = {k: str(k) for k in keys}
tree_map = dict(sorted(hash_map.items()))
access_order = OrderedDict(hash_map)
access_order.move_to_end(1)  # 'get' in an access-ordered LinkedHashMap
print("insertion:", list(hash_map), "| sorted:", list(tree_map), "| access-order:", list(access_order))

insertion: [5, 1, 9, 3] | sorted: [1, 3, 5, 9] | access-order: [5, 9, 3, 1]


## 13.6 Object Reflection
Reflection lets you inspect and invoke classes, methods, and fields at runtime, which is useful for debuggers, serializers, test frameworks, and dependency injection. Python is reflective by default.

In [7]:
class Point:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y

    def dist2(self) -> int:
        return self.x ** 2 + self.y ** 2


cls = globals()["Point"]                            # Class.forName("Point")
params = list(inspect.signature(cls).parameters)    # getConstructor().getParameters()
obj = cls(*[3, 4])                                  # newInstance(...)
method = getattr(obj, "dist2")                      # getMethod("dist2")
assert params == ["x", "y"] and method() == 25
assert [n for n, _ in inspect.getmembers(obj, inspect.ismethod)] == ["__init__", "dist2"]
setattr(obj, "x", 0)                                # field.set(obj, 0)
print(vars(obj))

{'x': 0, 'y': 4}


## 13.7 Lambda Expressions
There's a `Country` class with `continent` and `population` fields. Given a list of countries, compute the total population of one continent. Java uses streams and lambdas. Python uses a generator expression, or `filter`/`map` if you want to mirror Java.

In [8]:
from dataclasses import dataclass


@dataclass
class Country:
    name: str
    continent: str
    population: int


countries = [Country("A", "Europe", 10), Country("B", "Asia", 50), Country("C", "Europe", 7)]


def population(countries: list[Country], continent: str) -> int:
    return sum(c.population for c in countries if c.continent == continent)


def population_functional(countries: list[Country], continent: str) -> int:
    return sum(map(lambda c: c.population, filter(lambda c: c.continent == continent, countries)))


assert population(countries, "Europe") == population_functional(countries, "Europe") == 17
print("13.7 ok")

13.7 ok


## 13.8 Lambda Random
Using lambdas, write `get_random_subset(list)` that returns a random subset of any size, where every subset (including the empty one) is equally likely. Include each element independently with probability 1/2.

In [9]:
from collections import Counter


def get_random_subset(xs: list[int]) -> list[int]:
    return list(filter(lambda _: random.random() < 0.5, xs))


counts = Counter(tuple(get_random_subset([1, 2, 3])) for _ in range(80_000))
assert len(counts) == 8 and all(8_500 < c < 11_500 for c in counts.values())
print(dict(counts))

{(2,): 10026, (): 10020, (1, 2, 3): 9984, (3,): 10031, (2, 3): 9976, (1, 2): 9864, (1,): 10037, (1, 3): 10062}
